# Tutorial 1 – Extracting New ARCEME Data Cubes

This notebook walks through the full process of generating new multi-source satellite
data cubes using the ARCEME pipeline. Each cube covers a 10 × 10 km area centred on
an extreme weather event site and contains four satellite data sources:

| Layer | Source | Resolution |
|---|---|---|
| Sentinel-2 L2A (13 bands + SCL) | CDSE | 10 m |
| Sentinel-1 RTC (VV, VH) | Planetary Computer | 10 m |
| Copernicus DEM GLO-30 | CDSE | 10 m (resampled) |
| ESA WorldCover 2020 | Planetary Computer | 10 m |

The temporal window is **±12 months** around the event date.

---
**Prerequisites**  
- Python 3.11+ with `uv` installed *or* Docker  
- Access to [CDSE](https://dataspace.copernicus.eu/) credentials (free registration)  
- A `.env` file at the repository root with `CDSE_USERNAME` and `CDSE_PASSWORD`

## 1. Environment setup

The project uses [uv](https://github.com/astral-sh/uv) for reproducible dependency management.
Run the following from the **repository root** (only once):

In [ ]:
# Install uv (skip if already installed)
# !curl -LsSf https://astral.sh/uv/install.sh | sh

# Sync all dependencies into a local virtual environment
import subprocess, sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("Repository root:", repo_root)

result = subprocess.run(["uv", "sync"], cwd=repo_root, capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else "(no output)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-300:])

In [ ]:
# Verify key packages are available
import importlib
required = ["xarray", "zarr", "pystac_client", "stackstac", "pyproj", "pandas"]
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg}")
    except ImportError:
        print(f"  ✗ {pkg}  — run 'uv sync' first")

## 2. Preparing the input locations CSV

The pipeline reads a CSV with one row per location. The required columns are:

| Column | Description | Example |
|---|---|---|
| `DisNo.` | Unique location identifier (used in output filenames) | `wocat_966_dhp_41959` |
| `longitude` | Decimal degrees (WGS 84) | `4.6132` |
| `latitude` | Decimal degrees (WGS 84) | `50.7778` |
| `start_date` | Event date (YYYY-MM-DD) – pipeline adds ±12 months | `2020-06-05` |
| `wocat_id` | WOCAT database identifier | `966` |
| `dhp_label` | DHP database identifier | `41959` |
| `country` | ISO 3166-1 alpha-2 | `BE` |

Below we inspect the existing ARCEME locations file and create a minimal test subset.

In [ ]:
import pandas as pd

locations_path = "../data/selection_eu_wocat_dhp_qdoy.csv"
df = pd.read_csv(locations_path)
print(f"Total locations: {len(df)}")
df.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df["longitude"], df["latitude"], c="steelblue", s=40, alpha=0.8, zorder=3)
for _, row in df.iterrows():
    ax.annotate(row["DisNo."].split("_")[1], (row["longitude"], row["latitude"]),
                fontsize=7, ha="center", va="bottom", alpha=0.7)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("ARCEME event locations")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Create a minimal 2-row CSV for a quick test run
test_csv_path = "/tmp/arceme_test_locations.csv"
df.head(2).to_csv(test_csv_path, index=False)
print(f"Test CSV written to: {test_csv_path}")
pd.read_csv(test_csv_path)

## 3. Configuring the pipeline

The pipeline is controlled by a single YAML file. Below we programmatically create
a configuration for the test subset. Adjust paths and credentials as needed.

In [ ]:
import yaml
from pathlib import Path

output_base = "/tmp/arceme_test_output"   # <-- change to your target directory

config = {
    "locations_csv": test_csv_path,
    "skip_existing": True,

    "spatial": {
        "edge_size": 10000,   # 10 km × 10 km
        "units": "m",
        "resolution": 10,     # 10 m pixel size
    },

    "temporal": {
        "increment_months": 12,   # months BEFORE event
        "decrement_months": 12,   # months AFTER event
    },

    # Data sources: 'cdse' (Copernicus Data Space) or 'planetary' (Microsoft PC)
    "sources": {
        "s2":     "cdse",
        "s1":     "planetary",
        "copdem": "cdse",
        "esalc":  "planetary",
    },

    "collections": {
        "s2":     ["sentinel-2-l2a"],
        "s1":     ["sentinel-1-rtc"],
        "copdem": ["cop-dem-glo-30-dged-cog"],
        "esalc":  ["esa-worldcover"],
    },

    "static_dates": {
        "copdem": {"start": "2010-01-01", "end": "2024-12-31"},
        "esalc":  {"start": "2019-01-01", "end": "2020-12-31"},
    },

    "output_base_dir": output_base,

    "s2_filter_contains_bbox": False,

    "cloud_mask": {
        "enabled": True,
        "device": "cpu",   # change to 'cuda' if a GPU is available
        "stride": 512,
    },

    "merge": {
        "enabled": True,
        "chunk_time": 25,
        "chunk_x": 500,
        "chunk_y": 500,
        "prefer_latest_aux": True,
        "vars_to_uint16": [
            "B01","B02","B03","B04","B05","B06","B07",
            "B08","B8A","B09","B11","B12","SCL","cloud_mask","ESA_LC"
        ],
        "attrs": {
            "project": "ARCEME - Adaptation and Resilience to Climate Extremes and Multi-hazard Events",
            "data_cubes_producer": "CloudFerro S.A.",
        },
    },
}

config_path = "/tmp/arceme_test_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"Config written to: {config_path}")
print(yaml.dump(config, default_flow_style=False))

## 4. Running the pipeline

### Option A – native uv (recommended)

From the repository root, run:

```bash
# For long runs, start a tmux session first:
tmux new -s arceme_run

# Then launch the pipeline and tee output to a log file:
cd /path/to/data-cubes-arceme
uv run python src/processor/pipeline_orchestrator.py \
    --config /tmp/arceme_test_config.yaml \
    |& tee /tmp/arceme_test_output/run.log

# Detach from tmux:  Ctrl+b  d
# Re-attach later:   tmux attach -t arceme_run
```

### Option B – Docker (no local Python setup needed)

```bash
# Build once
cd /path/to/data-cubes-arceme
docker build -t arceme-pipeline:latest .

# Run – mount the output directory and pass the config via env var
docker run --rm -it \
  --env-file .env \
  -e PIPELINE_CONFIG=/tmp/arceme_test_config.yaml \
  -v /tmp/arceme_test_config.yaml:/tmp/arceme_test_config.yaml:ro \
  -v /tmp/arceme_test_output:/tmp/arceme_test_output \
  arceme-pipeline:latest \
  |& tee /tmp/arceme_test_output/run_docker.log
```

> **Tip**: for GPU-accelerated cloud masking add `--gpus all` to the `docker run` command
> and set `cloud_mask.device: cuda` in the config.

In [ ]:
# Optional: launch a short test run directly from this notebook.
# WARNING: this will download real satellite data and may take 20–60 min per location.
# Uncomment to execute.

# import subprocess, sys
# repo_root = os.path.abspath("..")
# result = subprocess.run(
#     ["uv", "run", "python", "src/processor/pipeline_orchestrator.py",
#      "--config", config_path],
#     cwd=repo_root,
#     text=True
# )
# print(result.returncode)

## 5. Monitoring pipeline progress

The pipeline logs one line per stage per location. You can monitor it with
`tail -f`, or use the helper below to count completed cubes across all stages.

In [ ]:
import os

stages = {
    "Sentinel-2 L2A":       "S2L2A",
    "S2 Cloud mask":        "S2L2A_CLOUDMASK",
    "Sentinel-1 RTC":       "S1RTC",
    "Copernicus DEM":        "COPDEM",
    "ESA WorldCover":        "ESALC",
    "Merged cubes (final)": "MERGED",
}

def count_zarr(directory: str) -> int:
    if not os.path.isdir(directory):
        return -1  # directory not yet created
    return sum(1 for e in os.scandir(directory) if e.name.endswith(".zarr"))

print(f"{'Stage':<25} {'Count':>6}  {'Directory'}")
print("-" * 70)
for label, subdir in stages.items():
    path = os.path.join(output_base, subdir)
    n = count_zarr(path)
    status = f"{n:>6}" if n >= 0 else "  (not created yet)"
    print(f"{label:<25} {status}  {path}")

In [ ]:
# Tail the last 30 lines of the log file (if it exists)
log_path = os.path.join(output_base, "run.log")
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print("".join(lines[-30:]))
else:
    print(f"Log file not found: {log_path}")

## 6. Verifying outputs

Once the pipeline finishes, the final merged cubes are in `{output_base}/MERGED/`.
Each zarr filename encodes the location and time window:

```
DC__{location}__{start_date}__{end_date}_{timestamp}_v0100.zarr
```

In [ ]:
import xarray as xr

merged_dir = os.path.join(output_base, "MERGED")
zarr_files = sorted([
    f for f in os.listdir(merged_dir) if f.endswith(".zarr")
]) if os.path.isdir(merged_dir) else []

print(f"Found {len(zarr_files)} merged cube(s) in {merged_dir}")
for name in zarr_files:
    print(" ", name)

In [ ]:
# Open the first cube (if any) and show its structure
if zarr_files:
    cube_path = os.path.join(merged_dir, zarr_files[0])
    ds = xr.open_zarr(cube_path)

    print("=== Dataset overview ===")
    print(ds)
    print()
    print("=== Global attributes ===")
    for k, v in ds.attrs.items():
        print(f"  {k}: {v}")
else:
    print("No merged cubes found. Run the pipeline first (Section 4).")

In [ ]:
# Quick sanity plot: count valid (non-NaN) Sentinel-2 observations over time
if zarr_files:
    valid_counts = (
        ds["B04"]
        .isnull()
        .pipe(lambda x: ~x)          # True where valid
        .sum(dim=["x", "y"])          # count valid pixels per time step
        .compute()
    )
    import pandas as pd
    t = pd.DatetimeIndex(ds["time_sentinel_2_l2a"].values)
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.bar(t, valid_counts, width=2, color="steelblue", alpha=0.8)
    ax.set_xlabel("Date")
    ax.set_ylabel("Valid pixels (B04)")
    ax.set_title("Valid Sentinel-2 observations over time")
    plt.tight_layout()
    plt.show()

## Summary

You have:
1. Prepared a locations CSV with event coordinates and dates
2. Written a `pipeline_config.yaml` controlling spatial, temporal and source settings
3. Launched the pipeline with `uv` or Docker
4. Verified merged zarr cubes in the output directory

Continue with **Tutorial 2** to open these cubes and perform footprint analysis,
RGB visualisation, cloud filtering, and NDVI anomaly computation.